# 03b — NHS Hospital Locations Audit

**Purpose:** Audit hospital locations, geocode via Code-Point Open, validate spatial
distribution, and produce a clean geocoded hospital dataset for healthcare accessibility
analysis (Layer 6).

**Input:** `data/raw/poi/hospitals.csv` — downloaded via NHS ORD 2-0-0 API (RO198,
England, HOSPITAL/INFIRMARY filter)
**Expected:** 3,870 rows, 4 columns (OrgId, Name, PostCode, LastChangeDate)
**Dependencies:** `data/audit/postcode_lookup.parquet` (Code-Point Open lookup)
**Output:** `data/audit/hospitals_geocoded.parquet`

In [1]:
import re

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger
from pathlib import Path
from pyproj import Transformer

ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
HOSPITALS_PATH = ROOT / "data/raw/poi/hospitals.csv"
POSTCODE_LOOKUP_PATH = ROOT / "data/audit/postcode_lookup.parquet"
OUTPUT_PATH = ROOT / "data/audit/hospitals_geocoded.parquet"
FIG_PATH = ROOT / "data/audit/fig_03b_hospitals.png"

checks: list[tuple[str, bool, str]] = []

## 1. Load and Profile hospitals.csv

In [2]:
df = pd.read_csv(HOSPITALS_PATH, skiprows=14)
logger.info(f"Loaded hospitals.csv: {df.shape[0]} rows x {df.shape[1]} cols")

print("=== Column inventory ===")
for col in df.columns:
    n_null = df[col].isna().sum()
    dtype = df[col].dtype
    if dtype == object:
        n_unique = df[col].nunique()
        sample = df[col].dropna().iloc[0] if n_null < len(df) else "N/A"
        print(f"  {col!r}: dtype={dtype}, nulls={n_null}, unique={n_unique}, sample={sample!r}")
    else:
        print(f"  {col!r}: dtype={dtype}, nulls={n_null}, min={df[col].min()}, max={df[col].max()}")

print("\n=== Head ===")
print(df.head(5).to_string())

2026-03-13 09:11:03.541 | INFO     | __main__:<module>:2 - Loaded hospitals.csv: 3870 rows x 4 cols


=== Column inventory ===
  'OrgId': dtype=str, nulls=0, min=A0E3V, max=Z9S7V
  'Name': dtype=str, nulls=0, min=A & E VICTORIA HOSPITAL, max=ZACHARY MERTON HOSPITAL
  'PostCode': dtype=str, nulls=0, min=AL10 9UA, max=YO8 9BX
  'LastChangeDate': dtype=str, nulls=0, min=2012-05-15, max=2026-03-10

=== Head ===
   OrgId                        Name  PostCode LastChangeDate
0  A0E3V  ROYAL ORTHOPAEDIC HOSPITAL   B31 2AP     2024-05-08
1  A0F9W               SKIN HOSPITAL   B71 4HJ     2024-05-08
2  A0H3E     COMMUNITY HOSPITAL WARD  BA11 2FH     2025-10-27
3  A0J8A        WEST MENDIP HOSPITAL   BA6 8JD     2024-05-08
4  A0T6V      FAIRFIELD GEN HOSPITAL   BL9 7TD     2024-05-08


## 2. Validate: row count, OrgId uniqueness, postcode format

In [3]:
# Row count
expected_rows = 3_870
row_count_ok = len(df) == expected_rows
checks.append(("Row count = 3,870", row_count_ok, f"Got {len(df)}"))
logger.info(f"Row count: {len(df)} -- {'PASS' if row_count_ok else 'FAIL'}")

# OrgId uniqueness
n_unique_orgid = df["OrgId"].nunique()
orgid_unique_ok = n_unique_orgid == len(df)
checks.append(("OrgId all unique", orgid_unique_ok, f"Unique={n_unique_orgid}, Total={len(df)}"))
logger.info(f"OrgId uniqueness: {n_unique_orgid}/{len(df)} -- {'PASS' if orgid_unique_ok else 'FAIL'}")

# Postcode format: [A-Z]{1,2}[0-9][0-9A-Z]? [0-9][A-Z]{2}
PC_REGEX = re.compile(r"^[A-Z]{1,2}[0-9][0-9A-Z]? [0-9][A-Z]{2}$")
n_null_pc = df["PostCode"].isna().sum()
valid_pc = df["PostCode"].dropna().apply(lambda x: bool(PC_REGEX.match(str(x).strip().upper())))
n_valid_pc = valid_pc.sum()
n_total_nonnull = len(valid_pc)
pc_format_ok = n_valid_pc == n_total_nonnull and n_null_pc == 0
checks.append((
    "All postcodes match standard format (zero nulls)",
    pc_format_ok,
    f"Valid={n_valid_pc}/{n_total_nonnull}, Nulls={n_null_pc}",
))
logger.info(f"Postcode format: {n_valid_pc}/{n_total_nonnull} valid, {n_null_pc} nulls -- {'PASS' if pc_format_ok else 'FAIL'}")

if not pc_format_ok:
    bad_mask = ~valid_pc.reindex(df.index, fill_value=False)
    bad = df[bad_mask]
    print("\nInvalid/null postcodes:")
    print(bad[["OrgId", "Name", "PostCode"]].to_string())

2026-03-13 09:11:03.551 | INFO     | __main__:<module>:5 - Row count: 3870 -- PASS


2026-03-13 09:11:03.552 | INFO     | __main__:<module>:11 - OrgId uniqueness: 3870/3870 -- PASS


2026-03-13 09:11:03.554 | INFO     | __main__:<module>:25 - Postcode format: 3870/3870 valid, 0 nulls -- PASS


## 3. Geocode via Code-Point Open postcode lookup

In [4]:
def standardise_postcode(pc):
    if pd.isna(pc):
        return None
    pc = re.sub(r"\s+", "", str(pc).strip().upper())
    return pc[:-3] + " " + pc[-3:] if len(pc) > 3 else pc


postcode_lookup = pd.read_parquet(POSTCODE_LOOKUP_PATH)
logger.info(f"Loaded postcode lookup: {len(postcode_lookup):,} postcodes")

df["postcode_std"] = df["PostCode"].apply(standardise_postcode)

# Merge
merged = df.merge(postcode_lookup, left_on="postcode_std", right_on="Postcode", how="left")
n_matched = merged["Easting"].notna().sum()
match_rate = n_matched / len(merged) * 100
match_ok = match_rate > 95.0
checks.append(("Postcode match rate > 95%", match_ok, f"Matched={n_matched}/{len(merged)} ({match_rate:.2f}%)"))
logger.info(f"Postcode match rate: {match_rate:.2f}% ({n_matched}/{len(merged)}) -- {'PASS' if match_ok else 'FAIL'}")

# Investigate unmatched
unmatched = merged[merged["Easting"].isna()][["OrgId", "Name", "PostCode", "postcode_std"]]
if len(unmatched) > 0:
    logger.warning(f"{len(unmatched)} unmatched postcodes:")
    print(unmatched.to_string())
else:
    logger.info("All postcodes matched -- zero unmatched")

2026-03-13 09:11:03.626 | INFO     | __main__:<module>:9 - Loaded postcode lookup: 1,492,016 postcodes


2026-03-13 09:11:03.809 | INFO     | __main__:<module>:19 - Postcode match rate: 95.97% (3714/3870) -- PASS


2026-03-13 09:11:03.811 | WARNING  | __main__:<module>:24 - 156 unmatched postcodes:


      OrgId                                                                            Name  PostCode postcode_std
114   B6D5N                                                 BUCKLAND COMMUNITY HOSPITAL CDC  CT17 0HD     CT17 0HD
115   B6Q2K                                                       SHREWSBURY COURT HOSPITAL   RH1 6YY      RH1 6YY
126   C1C6K                                                      ESPERANCE PRIVATE HOSPITAL  BN21 3BG     BN21 3BG
325   L2A5X  CAMBRIDGE UNIVERSITY HOSPITALS NHS FOUNDATION TRUST - COV BOOST COVID19 TRIALS   CB2 0AU      CB2 0AU
326   L2N2J                                                            HEATHERWOOD HOSPITAL   SL5 8AA      SL5 8AA
551   R1CF2                                                              ST JAMES' HOSPITAL   PO4 8LD      PO4 8LD
564   R1CT7                                                                FENWICK HOSPITAL  SO43 7NG     SO43 7NG
565   R1D01                                                                SHELT

## 4. Convert Easting/Northing (BNG EPSG:27700) -> Lat/Lon (WGS84 EPSG:4326)

In [5]:
transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=False)

# Only transform matched rows
mask = merged["Easting"].notna()
lats, lons = transformer.transform(
    merged.loc[mask, "Easting"].values,
    merged.loc[mask, "Northing"].values,
)
merged.loc[mask, "lat"] = lats
merged.loc[mask, "lon"] = lons

logger.info(f"Converted {mask.sum()} rows Easting/Northing -> lat/lon")
print(f"\nLat range: {merged['lat'].min():.4f} -- {merged['lat'].max():.4f}")
print(f"Lon range: {merged['lon'].min():.4f} -- {merged['lon'].max():.4f}")

2026-03-13 09:11:03.867 | INFO     | __main__:<module>:12 - Converted 3714 rows Easting/Northing -> lat/lon



Lat range: 49.9131 -- 55.4109
Lon range: -6.3093 -- 1.7300


## 5. Spatial validation

In [6]:
ENG_LAT_MIN, ENG_LAT_MAX = 49.8, 55.9
ENG_LON_MIN, ENG_LON_MAX = -6.5, 1.8

geocoded = merged[merged["lat"].notna()].copy()

lat_ok_mask = geocoded["lat"].between(ENG_LAT_MIN, ENG_LAT_MAX)
lon_ok_mask = geocoded["lon"].between(ENG_LON_MIN, ENG_LON_MAX)
n_outside_bounds = (~(lat_ok_mask & lon_ok_mask)).sum()

bounds_ok = n_outside_bounds == 0
checks.append((
    "All geocoded hospitals within England bounding box",
    bounds_ok,
    f"Outside bounds: {n_outside_bounds}",
))
logger.info(f"Spatial bounds check: {n_outside_bounds} outside England bbox -- {'PASS' if bounds_ok else 'FAIL'}")

if n_outside_bounds > 0:
    print("\nOut-of-bounds hospitals:")
    print(geocoded[~(lat_ok_mask & lon_ok_mask)][["OrgId", "Name", "PostCode", "lat", "lon"]].to_string())

# --- Scatter plot + Regional distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Scatter map
axes[0].scatter(geocoded["lon"], geocoded["lat"], s=4, alpha=0.5, color="#2563EB", linewidths=0)
axes[0].set_title(f"NHS Hospital Locations -- England (n={len(geocoded):,})", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_xlim(ENG_LON_MIN - 0.2, ENG_LON_MAX + 0.2)
axes[0].set_ylim(ENG_LAT_MIN - 0.2, ENG_LAT_MAX + 0.2)
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

# Regional distribution by postcode area (first 1-2 letters)
geocoded = geocoded.copy()
geocoded["pc_area"] = geocoded["PostCode"].str.extract(r"^([A-Z]{1,2})")
area_counts = geocoded["pc_area"].value_counts().sort_values(ascending=False).head(30)
axes[1].barh(area_counts.index[::-1], area_counts.values[::-1], color="#2563EB", alpha=0.85)
axes[1].set_title("Top 30 Postcode Areas by Hospital Count", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Number of hospitals")
axes[1].set_ylabel("Postcode area")
axes[1].tick_params(axis="y", labelsize=8)
axes[1].grid(axis="x", alpha=0.3)

plt.suptitle("03b -- NHS Hospital Locations Audit", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_PATH, dpi=150, bbox_inches="tight")
plt.close()
logger.info(f"Figure saved: {FIG_PATH}")

print(f"\nFull postcode area distribution ({len(area_counts)} shown of {geocoded['pc_area'].nunique()}):")
print(area_counts.to_string())

2026-03-13 09:11:03.875 | INFO     | __main__:<module>:16 - Spatial bounds check: 0 outside England bbox -- PASS


2026-03-13 09:11:04.185 | INFO     | __main__:<module>:50 - Figure saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/fig_03b_hospitals.png



Full postcode area distribution (30 shown of 98):
pc_area
B     132
NE    109
GU     84
GL     81
CM     74
BH     73
SO     72
L      72
TN     71
S      71
YO     70
CV     69
RH     65
BN     64
KT     62
SE     60
BS     60
SW     59
OX     59
EX     55
M      55
RG     55
LS     52
HA     47
W      46
PL     46
DE     46
WA     46
PE     46
NR     45


## 6. Name analysis -- hospital type categorisation

In [7]:
def classify_hospital(name: str) -> str:
    """Classify hospital by type based on Name keyword matching."""
    name_upper = str(name).upper()
    if "INFIRMARY" in name_upper:
        return "Infirmary"
    if "COMMUNITY" in name_upper:
        return "Community"
    if "WALK-IN" in name_upper or "WALK IN" in name_upper:
        return "Walk-in Centre"
    if "CENTRE" in name_upper or "CENTER" in name_upper:
        return "Centre"
    if "CLINIC" in name_upper:
        return "Clinic"
    if "UNIT" in name_upper:
        return "Unit"
    if "COTTAGE" in name_upper:
        return "Cottage Hospital"
    if "GENERAL" in name_upper or "GEN " in name_upper:
        return "General Hospital"
    if "ROYAL" in name_upper:
        return "Royal Hospital"
    if "HOSPITAL" in name_upper:
        return "Hospital"
    return "Other"


geocoded["hospital_type"] = geocoded["Name"].apply(classify_hospital)
type_breakdown = geocoded["hospital_type"].value_counts()
print("\n=== Hospital type breakdown ===")
print(type_breakdown.to_string())
logger.info(f"Hospital type categories: {geocoded['hospital_type'].nunique()}")

2026-03-13 09:11:04.193 | INFO     | __main__:<module>:31 - Hospital type categories: 9



=== Hospital type breakdown ===
hospital_type
Hospital            2600
Community            361
General Hospital     219
Royal Hospital       209
Infirmary            111
Clinic                74
Centre                60
Unit                  42
Cottage Hospital      38


## 7. Save hospitals_geocoded.parquet

In [8]:
output_cols = ["OrgId", "Name", "PostCode", "postcode_std", "LastChangeDate",
               "Easting", "Northing", "lat", "lon", "pc_area", "hospital_type"]
output_df = geocoded[output_cols].copy().reset_index(drop=True)
output_df.to_parquet(OUTPUT_PATH, index=False)
logger.info(f"Saved: {OUTPUT_PATH} -- {len(output_df):,} rows x {len(output_df.columns)} cols")

n_geocoded = len(output_df)
save_ok = OUTPUT_PATH.exists() and n_geocoded > 0
checks.append(("Output parquet saved successfully", save_ok, f"Rows: {n_geocoded}"))

2026-03-13 09:11:04.207 | INFO     | __main__:<module>:5 - Saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/hospitals_geocoded.parquet -- 3,714 rows x 11 cols


## 8. Update figures-registry.md -- GT-021

In [9]:
FIGURES_REGISTRY = ROOT / "docs/figures-registry.md"
registry_text = FIGURES_REGISTRY.read_text()

old_line = "| GT-021 | England acute hospital sites (geocoded) | 3,870 | \u2705 Confirmed | 03b_hospitals.ipynb | NHS ODS RO198, England only, HOSPITAL/INFIRMARY filter; match rate via Code-Point Open |"
new_line = f"| GT-021 | England acute hospital sites (geocoded) | {n_geocoded:,} | \u2705 Confirmed | 03b_hospitals.ipynb | NHS ODS RO198, England only, HOSPITAL/INFIRMARY filter; match rate via Code-Point Open ({match_rate:.1f}% geocoded) |"

if old_line in registry_text:
    updated_text = registry_text.replace(old_line, new_line)
    FIGURES_REGISTRY.write_text(updated_text)
    logger.info(f"Updated GT-021 in figures-registry.md: count={n_geocoded}, match_rate={match_rate:.1f}%")
    registry_ok = True
else:
    logger.warning("GT-021 line not found verbatim -- checking for partial match")
    # Try to find and update any GT-021 line
    lines = registry_text.splitlines()
    updated_lines = []
    registry_ok = False
    for line in lines:
        if "GT-021" in line and "England acute hospital sites" in line:
            updated_lines.append(new_line)
            registry_ok = True
            logger.info("Updated GT-021 via partial match")
        else:
            updated_lines.append(line)
    if registry_ok:
        FIGURES_REGISTRY.write_text("\n".join(updated_lines) + "\n")
    else:
        logger.error("GT-021 not found in figures-registry.md -- manual update required")
        print(f"Replacement line:\n  {new_line}")

checks.append(("figures-registry.md GT-021 updated", registry_ok, f"Geocoded count={n_geocoded}, match rate={match_rate:.1f}%"))

2026-03-13 09:11:04.213 | INFO     | __main__:<module>:10 - Updated GT-021 in figures-registry.md: count=3714, match_rate=96.0%


## 9. Validation summary

In [10]:
print("\n" + "=" * 65)
print("VALIDATION SUMMARY -- 03b_hospitals")
print("=" * 65)

fail_count = 0
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        fail_count += 1
    print(f"  [{status}] {name}")
    if detail:
        print(f"         -> {detail}")

print("-" * 65)
print(f"  Total checks: {len(checks)}")
print(f"  FAILs:        {fail_count}")
print("=" * 65)

print(f"\nKey figures:")
print(f"  GT-021  Geocoded hospital count:  {n_geocoded:,}")
print(f"  GT-021  Postcode match rate:       {match_rate:.2f}%")
print(f"  Output: {OUTPUT_PATH}")

assert fail_count == 0, f"{fail_count} FAILs -- see summary above"
logger.success("All checks passed. Hospital geocoded dataset saved.")

2026-03-13 09:11:04.217 | SUCCESS  | __main__:<module>:25 - All checks passed. Hospital geocoded dataset saved.



VALIDATION SUMMARY -- 03b_hospitals
  [PASS] Row count = 3,870
         -> Got 3870
  [PASS] OrgId all unique
         -> Unique=3870, Total=3870
  [PASS] All postcodes match standard format (zero nulls)
         -> Valid=3870/3870, Nulls=0
  [PASS] Postcode match rate > 95%
         -> Matched=3714/3870 (95.97%)
  [PASS] All geocoded hospitals within England bounding box
         -> Outside bounds: 0
  [PASS] Output parquet saved successfully
         -> Rows: 3714
  [PASS] figures-registry.md GT-021 updated
         -> Geocoded count=3714, match rate=96.0%
-----------------------------------------------------------------
  Total checks: 7
  FAILs:        0

Key figures:
  GT-021  Geocoded hospital count:  3,714
  GT-021  Postcode match rate:       95.97%
  Output: /Users/souravamseekarmarti/Projects/aequitas/data/audit/hospitals_geocoded.parquet
